# CV — Baseline Training (native aspect, no artificial padding)

**This replaces the previous 512/224 black-Letterbox training comparison as the next experimental baseline.**

The previous experiments achieved high Validation scores, but both resolutions retained a strong class-correlated padding cue: HCC had 0% added padding and Hemangioma about 19.8%. The previous results therefore cannot establish that the model learned lesion appearance.

## Revised experiment

- Input: full ultrasound frame after EXIF/RGB handling and conservative solid-black edge-strip trimming.
- Resize: shorter side = 224; aspect ratio preserved.
- Padding: **none**.
- Center crop / direct square warp: none.
- Augmentation: none.
- Model: ImageNet-pretrained ResNet18, frozen backbone and trained classification head.
- Variable H×W tensors: `batch_size=1`, with gradient accumulation.
- BatchNorm running statistics are frozen (the earlier code called `model.train()` and unintentionally updated frozen-backbone BatchNorm statistics).
- Training loss: each patient receives equal total image weight, so patients with many images do not dominate simply by contributing more frames.
- Validation: report both image-level and patient-level metrics; patient probabilities are the mean of that patient's image probabilities.
- Best checkpoint: selected by patient-level Validation log loss.
- Test: not loaded, not decoded, and not evaluated.

**Residual confounding:** this removes the artificial black letterbox band but does not eliminate the near-perfect original aspect-ratio/class association. Global average pooling allows variable spatial sizes without padding, but it does not prove the network cannot infer aspect ratio. We must not describe the shortcut as solved.

The Test split was inspected in earlier exploratory audits, so it is not a never-seen dataset historically. This notebook nevertheless does not construct Test image paths or use Test pixels/labels for training, model selection, or evaluation.

## Historical Letterbox baseline (for context only — not rerun)

These are the previously observed Validation results under the old black-Letterbox transform:

| Metric | 512×512 Letterbox | 224×224 Letterbox |
|---|---:|---:|
| Best epoch | 20 | 14 |
| Validation loss | 0.0970 | 0.0815 |
| Accuracy | 97.93% | 98.57% |
| HCC sensitivity | 97.29% | 98.49% |
| Specificity | 98.65% | 98.65% |
| F1 | 98.03% | 98.64% |
| ROC-AUC | 99.81% | 99.64% |
| Training time | 25.34 min | 12.45 min |

These numbers are **not evidence that Letterbox is safe**: both runs shared the same class-dependent padding pattern. They also used a Validation split that had already been inspected repeatedly. Do not use these results to claim clinical performance or choose a final resolution.

## 1. Environment, data paths, and split checks

This notebook uses `/content/drive/MyDrive/LiverSira AI/Training/data` in Colab, while locating `src/liversira_ai` in the parent `Training` directory. It can also run locally from inside the project.

Only Train and Validation rows are used to construct datasets. Test paths are not constructed.

In [ ]:
import os
import sys
from pathlib import Path

# Colab layout requested by the project:
# /content/drive/MyDrive/LiverSira AI/Training/
# ├── data/
# ├── src/
# └── notebooks/
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LiverSira AI/Training")

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

if (COLAB_PROJECT_ROOT / "src" / "liversira_ai").is_dir():
    PROJECT_ROOT = COLAB_PROJECT_ROOT
    DATA_WORKDIR = PROJECT_ROOT / "data"
    if DATA_WORKDIR.is_dir():
        os.chdir(DATA_WORKDIR)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (current, *current.parents)
         if (p / "src" / "liversira_ai").is_dir()),
        None
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Project root not found. Expected Training/src/liversira_ai "
            "or /content/drive/MyDrive/LiverSira AI/Training/src/liversira_ai."
        )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from liversira_ai.common.paths import CV_PROCESSED_DIR

DATASET_ROOT = Path(CV_PROCESSED_DIR) / "patient_split_v1"
METADATA_PATH = DATASET_ROOT / "patient_split_metadata.csv"

print({
    "project_root": str(PROJECT_ROOT),
    "working_directory": str(Path.cwd()),
    "dataset_root": str(DATASET_ROOT),
    "metadata_exists": METADATA_PATH.is_file(),
})
if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Missing patient split metadata: {METADATA_PATH}")

In [ ]:
import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet18_Weights, resnet18
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

SEED = 42
HCC_INDEX = 0
HEMANGIOMA_INDEX = 1
CLASS_TO_INDEX = {"HCC": HCC_INDEX, "Hemangioma": HEMANGIOMA_INDEX}
INDEX_TO_CLASS = {value: key for key, value in CLASS_TO_INDEX.items()}

SHORT_SIDE = 224
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "input_short_side": SHORT_SIDE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
})

## 2. Shared deterministic preprocessing

The functions below intentionally match the revised `04_image_preprocessing.ipynb`:
- EXIF transpose and RGB conversion;
- remove only complete near-black strips at the outer frame;
- resize the shorter side to 224;
- preserve aspect ratio;
- no padding, no center crop, no stretching.

Keep these two notebook implementations synchronized if the preprocessing rule changes.

In [ ]:
# Load metadata, then filter BEFORE constructing paths.
metadata_all = pd.read_csv(METADATA_PATH)
required_columns = {
    "split", "label", "patient_group_id", "destination_rel_path"
}
missing_columns = required_columns - set(metadata_all.columns)
if missing_columns:
    raise KeyError(f"Missing metadata columns: {sorted(missing_columns)}")

train_df = metadata_all[metadata_all["split"] == "train"].copy()
val_df = metadata_all[metadata_all["split"] == "val"].copy()

train_df["path"] = train_df["destination_rel_path"].map(
    lambda value: DATASET_ROOT / Path(str(value).replace("\\", "/"))
)
val_df["path"] = val_df["destination_rel_path"].map(
    lambda value: DATASET_ROOT / Path(str(value).replace("\\", "/"))
)

expected_train = {"HCC": 930, "Hemangioma": 840}
expected_val = {"HCC": 332, "Hemangioma": 297}

assert train_df.groupby("label").size().to_dict() == expected_train
assert val_df.groupby("label").size().to_dict() == expected_val
assert train_df["patient_group_id"].notna().all()
assert val_df["patient_group_id"].notna().all()

train_patients = set(train_df["patient_group_id"])
val_patients = set(val_df["patient_group_id"])
assert train_patients.isdisjoint(val_patients), "Train/Validation patient leakage"

# Only Train/Validation paths are accessed.
missing_train = train_df.loc[~train_df["path"].map(Path.is_file), "path"].astype(str).tolist()
missing_val = val_df.loc[~val_df["path"].map(Path.is_file), "path"].astype(str).tolist()
if missing_train or missing_val:
    raise FileNotFoundError(
        f"Missing Train files: {len(missing_train)}; "
        f"missing Validation files: {len(missing_val)}"
    )

display(pd.DataFrame([
    {"split": "train", "images": len(train_df), "patients": train_df["patient_group_id"].nunique()},
    {"split": "validation", "images": len(val_df), "patients": val_df["patient_group_id"].nunique()},
]))
print({
    "test_rows_used": False,
    "test_file_paths_constructed": False,
    "train_val_patient_overlap": 0,
    "train_images": len(train_df),
    "validation_images": len(val_df),
})

In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import torch

# Conservative, deterministic policy:
# - apply EXIF orientation;
# - convert to RGB;
# - remove only complete near-black outer rows/columns;
# - resize the shorter side to 224 while preserving aspect ratio;
# - do not center-crop, stretch, or add letterbox padding.
SHORT_SIDE = 224
DARK_THRESHOLD = 3
DARK_EDGE_FRACTION = 0.999
MAX_TRIM_FRACTION_PER_EDGE = 0.02

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def trim_uniform_black_border(
    image,
    threshold=DARK_THRESHOLD,
    dark_fraction=DARK_EDGE_FRACTION,
    max_fraction=MAX_TRIM_FRACTION_PER_EDGE,
):
    """
    Remove only complete near-black strips touching the outer image frame.

    A row/column is removed only when at least 99.9% of its pixels are
    <= threshold. Fan-shaped black corners are not removed because they
    do not occupy a complete row/column. No lesion or FOV segmentation
    is attempted.
    """
    image = ImageOps.exif_transpose(image).convert("RGB")
    gray = np.asarray(image.convert("L"), dtype=np.uint8)
    h, w = gray.shape

    row_dark_fraction = (gray <= threshold).mean(axis=1)
    col_dark_fraction = (gray <= threshold).mean(axis=0)

    max_top = int(h * max_fraction)
    max_bottom = int(h * max_fraction)
    max_left = int(w * max_fraction)
    max_right = int(w * max_fraction)

    top = 0
    while top < max_top and row_dark_fraction[top] >= dark_fraction:
        top += 1

    bottom = 0
    while bottom < max_bottom and row_dark_fraction[h - 1 - bottom] >= dark_fraction:
        bottom += 1

    left = 0
    while left < max_left and col_dark_fraction[left] >= dark_fraction:
        left += 1

    right = 0
    while right < max_right and col_dark_fraction[w - 1 - right] >= dark_fraction:
        right += 1

    # Guard against accidentally collapsing an image.
    if h - top - bottom < 32 or w - left - right < 32:
        top = bottom = left = right = 0

    cropped = image.crop((left, top, w - right, h - bottom))

    info = {
        "original_width": w,
        "original_height": h,
        "trim_top": top,
        "trim_bottom": bottom,
        "trim_left": left,
        "trim_right": right,
        "trimmed_width": cropped.width,
        "trimmed_height": cropped.height,
        "trimmed_fraction": 1.0 - (
            cropped.width * cropped.height / (w * h)
        ),
        "aspect_ratio_before": w / h,
        "aspect_ratio_after": cropped.width / cropped.height,
        "padding_pixels": 0,
    }
    return cropped, info


def resize_short_side_no_pad(image, short_side=SHORT_SIDE):
    """Resize while preserving aspect ratio; never crop or pad."""
    w, h = image.size
    scale = short_side / min(w, h)
    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))
    return image.resize((new_w, new_h), resample=Image.Resampling.BILINEAR)


def preprocess_image(image_or_path, short_side=SHORT_SIDE):
    """The single deterministic preprocessing function used in both notebooks."""
    if isinstance(image_or_path, (str, Path)):
        with Image.open(image_or_path) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
            image.load()
    else:
        image = ImageOps.exif_transpose(image_or_path).convert("RGB")

    trimmed, info = trim_uniform_black_border(image)
    resized = resize_short_side_no_pad(trimmed, short_side=short_side)

    info.update({
        "output_width": resized.width,
        "output_height": resized.height,
        "output_aspect_ratio": resized.width / resized.height,
        "output_short_side": min(resized.size),
    })
    return resized, info


def image_to_normalized_tensor(image):
    """RGB PIL image -> float32 CHW tensor with ImageNet normalization."""
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
    mean = torch.tensor(IMAGENET_MEAN, dtype=tensor.dtype).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=tensor.dtype).view(3, 1, 1)
    tensor = (tensor - mean) / std
    if not torch.isfinite(tensor).all():
        raise ValueError("Preprocessing generated NaN or Inf values.")
    return tensor


def legacy_black_letterbox(image, target=224):
    """Reference-only display of the old black-letterbox transform."""
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    scale = min(target / w, target / h)
    resized = image.resize(
        (max(1, round(w * scale)), max(1, round(h * scale))),
        resample=Image.Resampling.BILINEAR,
    )
    canvas = Image.new("RGB", (target, target), (0, 0, 0))
    x = (target - resized.width) // 2
    y = (target - resized.height) // 2
    canvas.paste(resized, (x, y))
    content_ratio = (resized.width * resized.height) / (target * target)
    return canvas, 1.0 - content_ratio


## 3. Model and evaluation protocol

HCC is the positive class and has index 0. Confusion-matrix rows/columns are explicitly ordered `[HCC, Hemangioma]`, so:

- `TP = cm[0, 0]`
- `FN = cm[0, 1]`
- `FP = cm[1, 0]`
- `TN = cm[1, 1]`

Patient-level metrics are computed after averaging each patient's image-level HCC probabilities. Patient-level Validation log loss determines early stopping and best-checkpoint selection.

In [ ]:
class NativeAspectUltrasoundDataset(Dataset):
    def __init__(self, frame, sample_weights=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.sample_weights = (
            np.ones(len(self.frame), dtype=np.float32)
            if sample_weights is None
            else np.asarray(sample_weights, dtype=np.float32)
        )
        if len(self.sample_weights) != len(self.frame):
            raise ValueError("sample_weights length does not match dataset.")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        record = self.frame.iloc[index]

        image, info = preprocess_image(
            record["path"],
            short_side=SHORT_SIDE
        )
        tensor = image_to_normalized_tensor(image)

        label = CLASS_TO_INDEX[record["label"]]
        patient_id = str(record["patient_group_id"])
        weight = float(self.sample_weights[index])

        return (
            tensor,
            torch.tensor(label, dtype=torch.long),
            patient_id,
            torch.tensor(weight, dtype=torch.float32),
        )


# Equal total loss weight per patient:
# each image receives 1 / number_of_images_of_its_patient.
patient_image_counts = train_df.groupby("patient_group_id")["path"].transform("count")
raw_weights = 1.0 / patient_image_counts.to_numpy(dtype=np.float32)
train_sample_weights = raw_weights / raw_weights.mean()

train_dataset = NativeAspectUltrasoundDataset(
    train_df,
    sample_weights=train_sample_weights
)
val_dataset = NativeAspectUltrasoundDataset(val_df)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=generator,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

# Check that a variable-size sample can pass through the loader.
sample_images, sample_labels, sample_patients, sample_weights = next(iter(val_loader))
print({
    "validation_batch_shape": tuple(sample_images.shape),
    "label": INDEX_TO_CLASS[int(sample_labels.item())],
    "patient_group_id": sample_patients[0],
    "sample_weight": float(sample_weights.item()),
    "finite": bool(torch.isfinite(sample_images).all()),
})
assert sample_images.ndim == 4 and sample_images.shape[0] == 1
assert min(sample_images.shape[-2:]) == SHORT_SIDE

## 4. Training

Variable spatial dimensions require `batch_size=1`. Gradient accumulation of four images gives an effective optimizer update every four images. The ResNet18 BatchNorm layers remain in evaluation mode because the backbone is frozen.

A patient-balanced sample weight of `1 / number_of_train_images_for_that_patient` (normalized to mean 1) gives each Train patient equal total contribution to the loss while preserving the observed patient counts by class. This does not make the two classes equally weighted.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def build_model():
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)

    # Freeze the feature extractor; train only the new two-class head.
    for parameter in model.parameters():
        parameter.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(DEVICE)


def keep_frozen_batchnorm(model):
    # model.train() otherwise switches frozen BatchNorm layers to training mode
    # and updates running statistics, even though their weights are frozen.
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def compute_metrics(y_true, hcc_probabilities, loss=None):
    y_true = np.asarray(y_true, dtype=int)
    hcc_probabilities = np.asarray(hcc_probabilities, dtype=float)
    predictions = np.where(
        hcc_probabilities >= 0.5,
        HCC_INDEX,
        HEMANGIOMA_INDEX
    )

    # Explicit order: HCC=positive class (0), Hemangioma=negative class (1).
    cm = confusion_matrix(
        y_true,
        predictions,
        labels=[HCC_INDEX, HEMANGIOMA_INDEX]
    )
    tp, fn, fp, tn = cm.ravel()

    result = {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "hcc_sensitivity": float(recall_score(
            y_true, predictions, pos_label=HCC_INDEX, zero_division=0
        )),
        "specificity": float(recall_score(
            y_true, predictions, pos_label=HEMANGIOMA_INDEX, zero_division=0
        )),
        "precision": float(precision_score(
            y_true, predictions, pos_label=HCC_INDEX, zero_division=0
        )),
        "f1": float(f1_score(
            y_true, predictions, pos_label=HCC_INDEX, zero_division=0
        )),
        "roc_auc": float(roc_auc_score(
            y_true == HCC_INDEX, hcc_probabilities
        )) if len(np.unique(y_true)) == 2 else float("nan"),
        "pr_auc": float(average_precision_score(
            y_true == HCC_INDEX, hcc_probabilities
        )) if len(np.unique(y_true)) == 2 else float("nan"),
        "tp": int(tp),
        "fn": int(fn),
        "fp": int(fp),
        "tn": int(tn),
        "confusion_matrix": cm,
    }
    if loss is not None:
        result["loss"] = float(loss)
    return result


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_hcc_probabilities = []
    all_patient_ids = []

    with torch.no_grad():
        for images, labels, patient_ids, _weights in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)
            probabilities = torch.softmax(logits, dim=1)[:, HCC_INDEX]

            total_loss += float(loss.item()) * len(images)
            all_labels.extend(labels.cpu().numpy().tolist())
            all_hcc_probabilities.extend(probabilities.cpu().numpy().tolist())
            all_patient_ids.extend(list(patient_ids))

    image_loss = total_loss / len(loader.dataset)
    image_metrics = compute_metrics(
        all_labels,
        all_hcc_probabilities,
        loss=image_loss
    )

    # Aggregate multiple images to one observation per patient.
    patient_frame = pd.DataFrame({
        "patient_group_id": all_patient_ids,
        "label": all_labels,
        "hcc_probability": all_hcc_probabilities,
    })

    label_consistency = patient_frame.groupby("patient_group_id")["label"].nunique()
    assert (label_consistency == 1).all()

    patient_frame = (
        patient_frame
        .groupby("patient_group_id", as_index=False)
        .agg(
            label=("label", "first"),
            hcc_probability=("hcc_probability", "mean"),
            n_images=("hcc_probability", "size"),
        )
    )

    patient_labels = patient_frame["label"].to_numpy(dtype=int)
    patient_probabilities = patient_frame["hcc_probability"].to_numpy(dtype=float)

    clipped = np.clip(patient_probabilities, 1e-7, 1 - 1e-7)
    patient_binary_truth = (patient_labels == HCC_INDEX).astype(int)
    patient_log_loss = log_loss(
        patient_binary_truth,
        clipped,
        labels=[0, 1]
    )

    patient_metrics = compute_metrics(
        patient_labels,
        patient_probabilities,
        loss=patient_log_loss
    )

    return {
        "image": image_metrics,
        "patient": patient_metrics,
        "patient_predictions": patient_frame,
        "image_labels": np.asarray(all_labels, dtype=int),
        "image_probabilities": np.asarray(all_hcc_probabilities, dtype=float),
    }

In [ ]:
def train_baseline():
    set_seed()

    model = build_model()
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    train_criterion = nn.CrossEntropyLoss(reduction="none")
    val_criterion = nn.CrossEntropyLoss()

    history = []
    best_state = None
    best_epoch = None
    best_patient_log_loss = float("inf")
    stale_epochs = 0
    started = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        keep_frozen_batchnorm(model)
        optimizer.zero_grad(set_to_none=True)

        weighted_loss_sum = 0.0
        total_weight = 0.0
        progress = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Native-aspect | Epoch {epoch}/{MAX_EPOCHS}",
            leave=True
        )

        for step, (images, labels, _patients, sample_weights) in progress:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            sample_weights = sample_weights.to(DEVICE, non_blocking=True)

            logits = model(images)
            per_image_losses = train_criterion(logits, labels)
            weighted_loss = (per_image_losses * sample_weights).mean()

            (weighted_loss / GRADIENT_ACCUMULATION_STEPS).backward()

            weighted_loss_sum += float(
                (per_image_losses * sample_weights).sum().detach().item()
            )
            total_weight += float(sample_weights.sum().detach().item())

            is_last_batch = (step + 1) == len(train_loader)
            is_accumulation_boundary = (
                (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            )

            if is_accumulation_boundary or is_last_batch:
                # Correct the final partial accumulation group.
                remainder = (step + 1) % GRADIENT_ACCUMULATION_STEPS
                if is_last_batch and remainder:
                    correction = GRADIENT_ACCUMULATION_STEPS / remainder
                    for parameter in trainable_parameters:
                        if parameter.grad is not None:
                            parameter.grad.mul_(correction)

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            progress.set_postfix(weighted_loss=f"{weighted_loss.item():.4f}")

        train_loss = weighted_loss_sum / max(total_weight, 1e-12)
        validation = evaluate(model, val_loader, val_criterion)

        image_metrics = validation["image"]
        patient_metrics = validation["patient"]

        row = {
            "epoch": epoch,
            "train_patient_weighted_loss": train_loss,
            "val_image_loss": image_metrics["loss"],
            "val_patient_log_loss": patient_metrics["loss"],
            "val_image_accuracy": image_metrics["accuracy"],
            "val_image_hcc_sensitivity": image_metrics["hcc_sensitivity"],
            "val_image_specificity": image_metrics["specificity"],
            "val_patient_accuracy": patient_metrics["accuracy"],
            "val_patient_hcc_sensitivity": patient_metrics["hcc_sensitivity"],
            "val_patient_specificity": patient_metrics["specificity"],
            "val_patient_f1": patient_metrics["f1"],
            "val_patient_roc_auc": patient_metrics["roc_auc"],
            "val_patient_pr_auc": patient_metrics["pr_auc"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"train weighted loss={train_loss:.4f} | "
            f"val image loss={image_metrics['loss']:.4f} | "
            f"val patient log loss={patient_metrics['loss']:.4f} | "
            f"patient Acc={patient_metrics['accuracy']:.4f} | "
            f"HCC Sens={patient_metrics['hcc_sensitivity']:.4f} | "
            f"Specificity={patient_metrics['specificity']:.4f} | "
            f"Patient AUC={patient_metrics['roc_auc']:.4f}"
        )

        # Model selection uses patient-level Validation log loss.
        if patient_metrics["loss"] < best_patient_log_loss:
            best_patient_log_loss = patient_metrics["loss"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            best_validation = copy.deepcopy(validation)
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print("Early stopping triggered.")
                break

    elapsed = time.perf_counter() - started

    if best_state is None:
        raise RuntimeError("Training did not produce a valid checkpoint.")

    # Return the actual best model, not the last-epoch model.
    model.load_state_dict(best_state)
    model.eval()

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_patient_log_loss": best_patient_log_loss,
        "best_validation": best_validation,
        "seconds": elapsed,
        "train_images": len(train_dataset),
        "train_patients": train_df["patient_group_id"].nunique(),
        "val_images": len(val_dataset),
        "val_patients": val_df["patient_group_id"].nunique(),
        "preprocessing": {
            "short_side": SHORT_SIDE,
            "trim_uniform_black_border": True,
            "dark_threshold": DARK_THRESHOLD,
            "dark_edge_fraction": DARK_EDGE_FRACTION,
            "max_trim_fraction_per_edge": MAX_TRIM_FRACTION_PER_EDGE,
            "padding": "none",
            "aspect_ratio": "preserved",
            "normalization": "ImageNet mean/std",
            "augmentation": "none",
        },
    }

In [ ]:
experiment = train_baseline()

display(experiment["history"].round(4))

best_image = experiment["best_validation"]["image"]
best_patient = experiment["best_validation"]["patient"]

print("Best epoch:", experiment["best_epoch"])
print("Training time (minutes):", round(experiment["seconds"] / 60, 2))

print("\nBest image-level Validation metrics:")
display(pd.DataFrame([{
    key: value for key, value in best_image.items()
    if key not in {"confusion_matrix"}
}]).round(4))

print("\nBest patient-level Validation metrics:")
display(pd.DataFrame([{
    key: value for key, value in best_patient.items()
    if key not in {"confusion_matrix"}
}]).round(4))

In [ ]:
history = experiment["history"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    history["epoch"],
    history["train_patient_weighted_loss"],
    label="Train patient-weighted loss"
)
axes[0].plot(
    history["epoch"],
    history["val_image_loss"],
    label="Validation image loss"
)
axes[0].plot(
    history["epoch"],
    history["val_patient_log_loss"],
    label="Validation patient log loss"
)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history["epoch"], history["val_patient_accuracy"], label="Patient accuracy")
axes[1].plot(history["epoch"], history["val_patient_hcc_sensitivity"], label="Patient HCC sensitivity")
axes[1].plot(history["epoch"], history["val_patient_specificity"], label="Patient specificity")
axes[1].plot(history["epoch"], history["val_patient_f1"], label="Patient F1")
axes[1].set_title("Patient-level Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

# Explicitly ordered [HCC, Hemangioma].
patient_predictions = experiment["best_validation"]["patient_predictions"]
patient_y = patient_predictions["label"].to_numpy(dtype=int)
patient_p = patient_predictions["hcc_probability"].to_numpy(dtype=float)
patient_pred = np.where(patient_p >= 0.5, HCC_INDEX, HEMANGIOMA_INDEX)

cm = confusion_matrix(
    patient_y,
    patient_pred,
    labels=[HCC_INDEX, HEMANGIOMA_INDEX]
)
display(pd.DataFrame(
    cm,
    index=["True HCC", "True Hemangioma"],
    columns=["Predicted HCC", "Predicted Hemangioma"]
))

print(classification_report(
    patient_y,
    patient_pred,
    labels=[HCC_INDEX, HEMANGIOMA_INDEX],
    target_names=["HCC", "Hemangioma"],
    zero_division=0
))

## 5. Validation results

Review image-level and patient-level metrics separately. Patient-level results are the primary view for this dataset because multiple frames from one patient are not independent clinical cases.

The Validation split has already been used in prior exploratory and baseline comparisons; its results are development evidence, not an untouched final estimate. Test remains unused by this notebook.

In [ ]:
# Persist only the actual best model checkpoint (not a report or plot).
SAVE_BEST_CHECKPOINT = True

if SAVE_BEST_CHECKPOINT:
    checkpoint_dir = PROJECT_ROOT / "models" / "cv" / "baseline_native_aspect_no_padding_v1"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "resnet18_best.pt"

    torch.save({
        "model_state_dict": experiment["model"].state_dict(),
        "architecture": "resnet18",
        "class_to_index": CLASS_TO_INDEX,
        "best_epoch": experiment["best_epoch"],
        "best_patient_log_loss": experiment["best_patient_log_loss"],
        "best_image_metrics": best_image,
        "best_patient_metrics": {
            key: value for key, value in best_patient.items()
            if key != "confusion_matrix"
        },
        "preprocessing": experiment["preprocessing"],
        "seed": SEED,
        "training_policy": {
            "backbone": "frozen",
            "batchnorm": "eval",
            "patient_balanced_image_loss": True,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "augmentation": False,
        },
    }, checkpoint_path)

    print("Best checkpoint saved to:", checkpoint_path)
else:
    print("Checkpoint saving disabled; best model remains in notebook memory.")

# Sanity checks: finite history, correct model and explicit Test exclusion.
numeric_history = experiment["history"].select_dtypes(include=[np.number]).to_numpy()
assert np.isfinite(numeric_history).all()
assert experiment["train_images"] == 1770
assert experiment["val_images"] == 629
assert experiment["train_patients"] == 521
assert experiment["val_patients"] == 149
assert experiment["preprocessing"]["padding"] == "none"
assert experiment["preprocessing"]["augmentation"] == "none"

print({
    "completed": True,
    "best_epoch": experiment["best_epoch"],
    "test_loaded": False,
    "test_evaluated": False,
    "augmentation": False,
    "patient_level_validation_reported": True,
    "checkpoint_saved": SAVE_BEST_CHECKPOINT,
})